In [1]:
!pip install gensim scikit-learn nltk pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 57.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import nltk
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
data = {
    "query": [
        "I forgot my student portal password",
        "Unable to access my college account",
        "My login is not working",
        "Account blocked after multiple attempts",

        "Fee payment failed but amount deducted",
        "Scholarship amount not received",
        "Issue with fee receipt",
        "Payment showing pending",

        "College app keeps crashing",
        "Portal not loading properly",
        "Error while submitting assignment",
        "Website is very slow",

        "How to update my profile details",
        "Where can I change my course settings",
        "How to edit personal information",
        "Information about college facilities"
    ],
    "label": [
        "Account","Account","Account","Account",
        "Finance","Finance","Finance","Finance",
        "Technical","Technical","Technical","Technical",
        "General","General","General","General"
    ]
}

df = pd.DataFrame(data)

In [7]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [8]:
# Stopwords
stop_words = set(stopwords.words('english'))

# Preprocessing function
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

# Apply preprocessing
df['tokens'] = df['query'].apply(preprocess)

# Output
print(df[['query', 'tokens']])

                                      query  \
0       I forgot my student portal password   
1       Unable to access my college account   
2                   My login is not working   
3   Account blocked after multiple attempts   
4    Fee payment failed but amount deducted   
5           Scholarship amount not received   
6                    Issue with fee receipt   
7                   Payment showing pending   
8                College app keeps crashing   
9               Portal not loading properly   
10        Error while submitting assignment   
11                     Website is very slow   
12         How to update my profile details   
13    Where can I change my course settings   
14         How to edit personal information   
15     Information about college facilities   

                                      tokens  
0        [forgot, student, portal, password]  
1         [unable, access, college, account]  
2                           [login, working]  
3     [accou

In [9]:
w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)


In [10]:

def sentence_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


In [11]:

df['vector'] = df['tokens'].apply(lambda x: sentence_vector(x, w2v_model))

X = np.vstack(df['vector'].values)
y = df['label']

In [12]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 0.5


In [13]:
def predict_query(query):
    tokens = preprocess(query)
    vec = sentence_vector(tokens, w2v_model).reshape(1, -1)
    return model.predict(vec)[0]


# Test with your dataset context
print(predict_query("I cannot access my student account"))
print(predict_query("Fee payment failed and money deducted"))
print(predict_query("College portal is not opening"))
print(predict_query("How to update my profile"))

Account
Finance
Technical
General
